### LCEL (대화내용 기억하기): 메모리 추가

In [1]:
!pip --version

pip 26.2.1 from d:\hanhwa0902\ex0918\.0918venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from operator import itemgetter
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

model = ChatOpenAI()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [4]:
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

C:\Users\user\AppData\Local\Temp\ipykernel_10304\2844409624.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")


In [5]:
memory.load_memory_variables({})

{'chat_history': []}

In [6]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")
)

In [7]:
runnable.invoke({"input": "hi"})

{'input': 'hi', 'chat_history': []}

In [8]:
runnable.invoke({"input": "hi"})

{'input': 'hi', 'chat_history': []}

In [9]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [10]:
runnable.invoke({"input": "hi!"})

{'input': 'hi!', 'chat_history': []}

In [11]:
chain = runnable | prompt | model

In [12]:
response = chain.invoke({"input": "만나서 반갑습니다. 제 이름은 테디입니다."})
print(response.content)

만나서 반가워요, 테디님! 무엇을 도와드릴까요?


In [13]:
memory.load_memory_variables({})

{'chat_history': []}

In [14]:
memory.save_context(
    {"human": "만나서 반갑습니다. 제 이름은 테디입니다."}, {"ai": response.content}
)

memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='만나서 반갑습니다. 제 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='만나서 반가워요, 테디님! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [15]:
response = chain.invoke({"input": "제 이름이 무엇이었는지 기억하세요?"})

print(response.content)

네, 테디님이세요! 부담 갖지 마세요. 어떤 것을 도와드릴까요?


#### Custom ConversationChain 구현 예시

In [16]:
from operator import itemgetter
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, Runnable
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

In [17]:
class MyConversationChain(Runnable):

    def __init__(self, llm, prompt, memory, input_key="input"):

        self.prompt = prompt
        self.memory = memory
        self.input_key = input_key

        self.chain = (
            RunnablePassthrough.assign(
                chat_history=RunnableLambda(self.memory.load_memory_variables)
                | itemgetter(memory.memory_key)
            )
            | prompt
            | llm
            | StrOutputParser()
        )

    def invoke(self, query, configs=None, **kwargs): #kwargs: keyword arguments #**kwargs: 이름=값 형태로 전달한 인자들을 함수 내부에서 딕셔너리 구조로 묶어서 처리
        answer = self.chain.invoke({self.input_key: query})
        self.memory.save_context(inputs={"human": query}, outputs={"ai": answer})
        return answer

In [20]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a hlepful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

conversation_chain = MyConversationChain(llm, prompt, memory)

In [21]:
conversation_chain.invoke("제 이름이 뭐라고요?")

'죄송하지만 지금은 사용자의 이름을 알 수 없어요. 저는 계정 정보나 개인 데이터에 접근할 권한이 없습니다. 이름을 알려주시면 그 이름으로 불러드릴게요. 어떻게 부르면 될까요? (원하시면 닉네임 추천도 해드려요.)'

In [22]:
conversation_chain.invoke("앞으로 제 이름은 테디입니다.")

'알겠어요, 테디님. 앞으로 그렇게 부를게요.  \n(이 대화에서는 기억하지만, 새 대화를 시작하면 다시 알려주셔야 할 수도 있어요.) 다른 호칭이나 별명 원하시면 말씀해 주세요.'

In [23]:
conversation_chain.memory.load_memory_variables({})["chat_history"]

[HumanMessage(content='제 이름이 뭐라고요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='죄송하지만 지금은 사용자의 이름을 알 수 없어요. 저는 계정 정보나 개인 데이터에 접근할 권한이 없습니다. 이름을 알려주시면 그 이름으로 불러드릴게요. 어떻게 부르면 될까요? (원하시면 닉네임 추천도 해드려요.)', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='앞으로 제 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='알겠어요, 테디님. 앞으로 그렇게 부를게요.  \n(이 대화에서는 기억하지만, 새 대화를 시작하면 다시 알려주셔야 할 수도 있어요.) 다른 호칭이나 별명 원하시면 말씀해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]